# 02 · Data preparation: windows and features (M4)

CRISP-DM data preparation for the LogTriage Agent. `logtriage prepare` turns parsed lines into one-minute windows with features, for `russellmitchell` (dev) and `santos` (test). This notebook only reads its outputs and shows them.

It reads no ground truth (§5.2): no labels, no attack phases and no attack timing, and `tests/test_isolation.py` checks that. Every number below comes from the window tables; the closing note of §7 adds context from D16, which M3 recorded.

In [1]:
import json
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # configured paths are relative to the repository root
    os.chdir("..")

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from logtriage.config import DEV_TESTBED, TEST_TESTBED, load_settings
from logtriage.data.features import (
    FEATURE_SETS,
    REPORT_FILE,
    SCALER_DIR,
    VOCABULARY_DIR,
    FeatureScaler,
    feature_summary,
)
from logtriage.data.periods import load_periods
from logtriage.data.store import connect

TESTBEDS = [DEV_TESTBED, TEST_TESTBED]
settings = load_settings()
directories = {name: settings.paths.processed_dir / name for name in TESTBEDS}
reports = {
    name: json.loads((path / REPORT_FILE).read_text(encoding="utf-8"))
    for name, path in directories.items()
}
windows = {
    name: {
        log_type: pd.read_parquet(path / f"windows_{log_type}.parquet") for log_type in FEATURE_SETS
    }
    for name, path in directories.items()
}


def show(title: str, frame: pd.DataFrame) -> None:
    """Display a table under a heading."""
    display(Markdown(f"**{title}**"))
    display(frame)

## 1. Periods

Boundaries come from `periods.json` (§4.3). Vocabularies, scalers and anomaly models are fit only on `fit`. A window takes the period in which it ends, so a `fit` window holds only fit lines (D24).

In [2]:
rows = []
for name, path in directories.items():
    periods = load_periods(path / "periods.json")
    rows += [
        {"testbed": name, "period": "fit", "start": periods.train_start, "end": periods.fit_end},
        {"testbed": name, "period": "calib", "start": periods.fit_end, "end": periods.train_end},
        {"testbed": name, "period": "eval", "start": periods.eval_start, "end": periods.eval_end},
    ]
table = pd.DataFrame(rows)
table["hours"] = ((table["end"] - table["start"]).dt.total_seconds() / 3600).round(1)
show("Periods (UTC)", table.set_index(["testbed", "period"]))

**Periods (UTC)**

start  \
testbed         period                                 
russellmitchell fit    2022-01-21 00:00:00.000+00:00   
                calib  2022-01-22 16:48:48.000+00:00   
                eval   2022-01-23 03:01:00.000+00:00   
santos          fit    2022-01-14 00:00:00.000+00:00   
                calib  2022-01-15 01:01:01.600+00:00   
                eval   2022-01-15 07:16:17.000+00:00   

                                                 end  hours  
testbed         period                                       
russellmitchell fit    2022-01-22 16:48:48.000+00:00   40.8  
                calib  2022-01-23 03:01:00.000+00:00   10.2  
                eval   2022-01-25 00:00:00.000+00:00   45.0  
santos          fit    2022-01-15 01:01:01.600+00:00   25.0  
                calib  2022-01-15 07:16:17.000+00:00    6.3  
                eval   2022-01-18 00:00:00.000+00:00   64.7

## 2. Windows per log type

A window is one `(testbed, host, log_type, minute)` with at least one parsed line. Lines with `parse_ok=False` stay in `lines_*.parquet` but are left out (D14), because their timestamps are inherited from another line. A log type is baseline-capable with at least `ml.min_fit_lines` parsed fit lines and `ml.min_calib_lines` parsed calib lines (D19).

In [3]:
with connect() as con:
    for name, path in directories.items():
        rows = []
        for log_type, entry in reports[name].items():
            lines = path / f"lines_{log_type}.parquet"
            total, parsed = con.execute(
                f"SELECT count(*), count(*) FILTER (parse_ok) FROM '{lines}'"
            ).fetchone()
            frame = windows[name][log_type]
            rows.append(
                {
                    "log_type": log_type,
                    "lines": total,
                    "parse_ok_false": total - parsed,
                    "lines_in_windows": int(frame["n_lines"].sum()),
                    "hosts": frame["host"].nunique(),
                    "fit_lines": entry["fit_lines"],
                    "calib_lines": entry["calib_lines"],
                    "fit_windows": entry["windows"]["fit"],
                    "calib_windows": entry["windows"]["calib"],
                    "eval_windows": entry["windows"]["eval"],
                    "baseline_capable": entry["baseline_capable"],
                }
            )
        show(f"{name}: windows", pd.DataFrame(rows).set_index("log_type"))

**russellmitchell: windows**

,lines,parse_ok_false,lines_in_windows,hosts,fit_lines,calib_lines,fit_windows,calib_windows,eval_windows,baseline_capable
log_type,,,,,,,,,,
apache_access,103337,0,103337,3,40217,7933,4172,1264,3458,True
apache_error,4858,4763,95,3,13,0,5,0,11,False
audit,18740,0,18740,7,8372,1384,1393,209,1514,True
auth,5319,0,5319,10,2283,544,1136,272,1249,True
dns,503498,0,503498,2,231943,34904,4896,1226,4855,True
vpn,5537,0,5537,1,2291,439,119,24,157,True


**santos: windows**

,lines,parse_ok_false,lines_in_windows,hosts,fit_lines,calib_lines,fit_windows,calib_windows,eval_windows,baseline_capable
log_type,,,,,,,,,,
apache_access,210121,0,210121,3,51221,1581,1832,83,5632,True
apache_error,4799,4723,76,3,0,10,0,3,9,False
audit,24042,0,24042,7,5956,1001,993,164,2730,True
auth,5314,0,5314,10,1375,352,692,176,1798,True
dns,531462,0,531462,2,146692,17898,3002,750,5949,True
vpn,5622,0,5622,1,1395,147,77,7,227,True


`lines_in_windows` equals `lines` minus `parse_ok_false` for every log type. Only `apache_error` is not baseline-capable, on both testbeds, as M3 found.

## 3. Dropped features

Two rules remove features (§6.3). Vocabulary, rarity and "unseen" features go when a log type has fewer than `ml.min_fit_lines` parsed fit lines (D15). Features with one value over all windows go as constant (D25). The check spans all periods, because `n_unseen_*` features are 0 on every fit window by construction.

In [4]:
rows = [
    {
        "testbed": name,
        "log_type": log_type,
        "features_kept": len(entry["features"]),
        "dropped_by_D15": ", ".join(entry["dropped_vocabulary"]) or "-",
        "dropped_as_constant": ", ".join(entry["dropped_constant"]) or "-",
    }
    for name, report in reports.items()
    for log_type, entry in report.items()
]
with pd.option_context("display.max_colwidth", None):
    show("Features kept and dropped", pd.DataFrame(rows).set_index(["testbed", "log_type"]))

**Features kept and dropped**

features_kept  \
testbed         log_type                       
russellmitchell apache_access             19   
                apache_error               4   
                audit                      9   
                auth                      14   
                dns                       15   
                vpn                       11   
santos          apache_access             18   
                apache_error               4   
                audit                      9   
                auth                      14   
                dns                       15   
                vpn                       11   

                                                                                                      dropped_by_D15  \
testbed         log_type                                                                                               
russellmitchell apache_access                                                                                      -   
                apache_error   n_unseen_templates, ratio_unseen_templates, mean_template_rarity, max_template_rarity   
                audit                                                                                              -   
                auth                                                                                               -   
                dns                                                                                                -   
                vpn                                                                                                -   
santos          apache_access                                                                                      -   
                apache_error   n_unseen_templates, ratio_unseen_templates, mean_template_rarity, max_template_rarity   
                audit                                                                                              -   
                auth                                                                                               -   
                dns                                                                                                -   
                vpn                                                                                                -   

                                                      dropped_as_constant  
testbed         log_type                                                   
russellmitchell apache_access                                           -  
                apache_error                                            -  
                audit          n_execve, n_execve_web_user, n_root_execve  
                auth                                                    -  
                dns                                                     -  
                vpn                                        n_unseen_users  
santos          apache_access                             n_unseen_src_ip  
                apache_error                                            -  
                audit          n_execve, n_execve_web_user, n_root_execve  
                auth                                                    -  
                dns                                                     -  
                vpn                                        n_unseen_users

The audit logs of both testbeds hold no `EXECVE` record, and every `SYSCALL` record has `syscall=1`, so all three execve features are constant at 0. `audit` keeps `n_unique_exe`, `n_unseen_exe` and `n_user_auth_events`.

## 4. Feature summary per log type

Raw values: the mean per period, the maximum over all windows and the share of windows with a non-zero value. `NaN` means the log type has no window in that period.

In [5]:
for name in TESTBEDS:
    display(Markdown(f"### {name}"))
    for log_type in FEATURE_SETS:
        show(
            f"{log_type} ({len(windows[name][log_type]):,} windows)",
            feature_summary(windows[name][log_type]),
        )

### russellmitchell

**apache_access (8,894 windows)**

period,mean_fit,mean_calib,mean_eval,max,share_nonzero
n_lines,9.639,6.279,15.959,4687.000,1.000
n_unseen_templates,0.000,0.003,2.238,4687.000,0.002
ratio_unseen_templates,0.000,0.000,0.002,1.000,0.002
mean_template_rarity,1.012,0.662,1.264,10.603,1.000
max_template_rarity,1.919,1.004,2.427,10.603,1.000
n_unique_templates,1.832,1.218,2.010,8.000,1.000
n_req,9.639,6.279,15.959,4687.000,1.000
ratio_4xx,0.007,0.001,0.011,1.000,0.063
ratio_404,0.002,0.000,0.002,0.992,0.010
ratio_5xx,0.000,0.000,0.000,0.000,0.000


**apache_error (16 windows)**

period,mean_fit,mean_calib,mean_eval,max,share_nonzero
n_lines,2.6,NaN,7.455,27,1.000
n_unique_templates,2.6,NaN,2.909,6,1.000
n_errors,0.2,NaN,5.455,27,0.312
n_unique_clients,0.2,NaN,0.545,2,0.375


**auth (2,657 windows)**

... (16 outputs left out)

**audit (3,887 windows)**

period,mean_fit,mean_calib,mean_eval,max,share_nonzero
n_lines,5.998,6.104,6.258,32.000,1.000
n_unseen_templates,0.000,0.000,0.012,15.000,0.001
ratio_unseen_templates,0.000,0.000,0.001,0.789,0.001
mean_template_rarity,1.981,2.300,1.936,7.285,1.000
max_template_rarity,2.253,2.611,2.210,8.694,1.000
n_unique_templates,4.407,5.738,4.297,11.000,1.000
n_unique_exe,1.232,1.323,1.225,4.000,1.000
n_unseen_exe,0.000,0.000,0.003,2.000,0.001
n_user_auth_events,2.212,1.860,2.421,16.000,0.948


**vpn (311 windows)**

period,mean_fit,mean_calib,mean_eval,max,share_nonzero
n_lines,18.117,21.000,17.974,56.000,1.000
n_unseen_templates,0.000,0.000,0.013,1.000,0.010
ratio_unseen_templates,0.000,0.000,0.003,0.500,0.010
mean_template_rarity,2.914,2.184,2.959,6.296,1.000
max_template_rarity,4.247,3.546,4.282,7.260,1.000
n_unique_templates,11.091,12.000,10.841,20.000,1.000
n_auth_success,0.247,0.000,0.256,2.000,0.244
n_auth_fail,0.026,0.000,0.013,1.000,0.016
n_new_sessions,0.247,0.000,0.256,2.000,0.244
n_unique_src_ip,1.026,1.000,1.022,2.000,1.000


## 5. Vocabularies

Built from fit-period lines only, per testbed and log type, and pooled across hosts. A test in `tests/test_features.py` changes every later line and checks that vocabularies and scalers stay the same.

In [6]:
rows = []
for name, path in directories.items():
    for file in sorted((path / VOCABULARY_DIR).glob("*.parquet")):
        table = pd.read_parquet(file)
        sizes = table.groupby("vocabulary").size()
        rows += [
            {"testbed": name, "log_type": file.stem, "vocabulary": key, "values": int(size)}
            for key, size in sizes.items()
        ]
show(
    "Vocabulary sizes",
    pd.DataFrame(rows)
    .pivot_table(index=["log_type", "vocabulary"], columns="testbed", values="values")
    .astype(int),
)

**Vocabulary sizes**

testbed                   russellmitchell  santos
log_type      vocabulary                         
apache_access paths                   541     680
              src_ip                    7      12
              ua                        6       6
audit         exe                       5       4
auth          users                     7       6
dns           domains                 618     736
vpn           src_ip                    3       3
              users                     3       3

## 6. Scaling

Counts get `log1p`, then a `RobustScaler` fit on the `fit` windows, pooled across hosts. Windows keep raw values; M5 applies the saved scaler (D26). Baseline-capable types have one scaler each. The median of a scaled fit window is 0 by construction, so the useful columns are the shares of later windows far from the fit range. All shares are unclipped except the last, which applies the one-sided clip of six DNS features (D29).

In [7]:
rows = []
for name, path in directories.items():
    for file in sorted((path / SCALER_DIR).glob("*.json")):
        scaler = FeatureScaler.load(file)
        frame = windows[name][file.stem]
        scaled = {
            period: scaler.transform(frame[frame["period"] == period], clip=False)
            for period in ("fit", "calib", "eval")
        }
        clipped = scaler.transform(frame[frame["period"] == "eval"])
        rows.append(
            {
                "testbed": name,
                "log_type": file.stem,
                "columns": len(scaler.columns),
                "fit_median_abs": float(np.abs(np.median(scaled["fit"], axis=0)).max()),
                "share_abs_over_3_fit": float((np.abs(scaled["fit"]) > 3).any(axis=1).mean()),
                "share_abs_over_3_calib": float((np.abs(scaled["calib"]) > 3).any(axis=1).mean()),
                "share_abs_over_3_eval": float((np.abs(scaled["eval"]) > 3).any(axis=1).mean()),
                "share_abs_over_3_eval_clipped": float((np.abs(clipped) > 3).any(axis=1).mean()),
            }
        )
show("Scaled windows", pd.DataFrame(rows).set_index(["testbed", "log_type"]).round(3))

**Scaled windows**

columns  fit_median_abs  share_abs_over_3_fit  \
testbed         log_type                                                       
russellmitchell apache_access       19             0.0                 0.361   
                audit                9             0.0                 0.007   
                auth                14             0.0                 0.018   
                dns                 15             0.0                 0.064   
                vpn                 11             0.0                 0.185   
santos          apache_access       18             0.0                 0.106   
                audit                9             0.0                 0.001   
                auth                14             0.0                 0.016   
                dns                 15             0.0                 0.026   
                vpn                 11             0.0                 0.494   

                               share_abs_over_3_calib  share_abs_over_3_eval  \
testbed         log_type                                                       
russellmitchell apache_access                   0.096                  0.486   
                audit                           0.000                  0.003   
                auth                            0.000                  0.018   
                dns                             0.021                  0.207   
                vpn                             0.208                  0.255   
santos          apache_access                   0.205                  0.119   
                audit                           0.000                  0.002   
                auth                            0.011                  0.019   
                dns                             0.015                  0.529   
                vpn                             0.000                  0.511   

                               share_abs_over_3_eval_clipped  
testbed         log_type                                      
russellmitchell apache_access                          0.486  
                audit                                  0.003  
                auth                                   0.018  
                dns                                    0.105  
                vpn                                    0.255  
santos          apache_access                          0.119  
                audit                                  0.002  
                auth                                   0.019  
                dns                                    0.094  
                vpn                                    0.511

## 7. DNS: what the qname features add

`n_unseen_templates` carries almost no DNS signal, because Drain3 masks the encoded names into templates seen in the fit period. The qname features read the names themselves.

In [8]:
columns = [
    "n_unseen_templates",
    "n_queries",
    "max_label_len",
    "max_label_entropy",
    "top_domain_share",
    "n_unseen_domains",
]
for name in TESTBEDS:
    frame = windows[name]["dns"]
    means = frame.groupby(["host", "period"])[columns].mean().round(3)
    show(f"{name}: DNS means per host and period", means)

**russellmitchell: DNS means per host and period**

n_unseen_templates  n_queries  max_label_len  \
host          period                                                 
inet-dns      calib                0.000      6.581         35.002   
              eval                 0.001     11.826         31.928   
              fit                  0.000     11.538         35.039   
inet-firewall calib                0.000     10.485         35.002   
              eval                 0.000     15.693         32.080   
              fit                  0.000     14.967         35.054   

                      max_label_entropy  top_domain_share  n_unseen_domains  
host          period                                                         
inet-dns      calib               4.880             0.853             0.008  
              eval                4.623             0.698             0.230  
              fit                 4.876             0.710             0.000  
inet-firewall calib               4.880             0.854             0.008  
              eval                4.622             0.732             0.222  
              fit                 4.876             0.727             0.000

**santos: DNS means per host and period**

n_unseen_templates  n_queries  max_label_len  \
host          period                                                 
inet-dns      calib                0.000      7.531         61.000   
              eval                 0.001     15.173         36.823   
              fit                  0.000     12.985         61.000   
inet-firewall calib                0.000      6.664         61.000   
              eval                 0.000     16.709         36.394   
              fit                  0.000     13.386         61.000   

                      max_label_entropy  top_domain_share  n_unseen_domains  
host          period                                                         
inet-dns      calib               5.329             0.859             0.053  
              eval                4.212             0.620             0.453  
              fit                 5.330             0.696             0.000  
inet-firewall calib               5.329             0.911             0.053  
              eval                4.131             0.667             0.478  
              fit                 5.330             0.744             0.000

Long, high-entropy labels and a dominant domain are the normal state of the `fit` period, not an exception. On santos, `max_label_len` averages 61 on `fit` and `calib` and 36 on `eval`, and `max_label_entropy` falls from 5.33 to 4.17. Russellmitchell moves the same way, less strongly: 35 to 32, and 4.88 to 4.62.

So the baseline is the unusual state. This matches D16: exfiltration runs through the whole training period, the models will learn it as normal, and the fit vocabulary holds its domain, so `n_unseen_domains` cannot flag it. It also has a second effect: windows after the traffic stops sit far from the fit range, which is why 52.9% of santos `dns` eval windows in §6 have an unclipped scaled value beyond 3. A value below this baseline cannot be the attack, so D29 clips six DNS features at 0 on the low side. The remainder in the last column of §6 is a known limitation, mostly `mean_template_rarity` after the tunnel stops. M5 and M11 report DNS results separately, as D16 requires.